# Evaluation Report — Baseline vs. ML-Enhanced IDS

This notebook generates publication-ready evaluation results:
1. Load test data and trained models
2. Run all models on test set
3. Compute metrics (Accuracy, Precision, Recall, F1, ROC-AUC, FPR)
4. Baseline vs. Enhanced comparison
5. Generate charts for publication

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

project_root = str(Path.cwd().parent) if 'notebooks' in str(Path.cwd()) else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.feature_config import *
from src.models.model_utils import *
from src.evaluation.metrics import compute_all_metrics, print_metrics_table, plot_roc_curves
from src.evaluation.compare import simulate_baseline

plt.style.use('seaborn-v0_8-darkgrid')
print('Setup complete!')

In [ ]:
# Load data and models
data = load_processed_data(os.path.join(project_root, 'data/processed'))
X_test, y_test = data['X_test'], data['y_test']
print(f'Test set: {X_test.shape}')

rf = load_sklearn_model(RF_MODEL_FILE)
xgb = load_sklearn_model(XGB_MODEL_FILE)
dnn = load_keras_model(DNN_MODEL_FILE)
meta = load_sklearn_model(META_MODEL_FILE)
print('All models loaded!')

In [ ]:
# Run predictions
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)
xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)
dnn_prob = dnn.predict(X_test, verbose=0)
dnn_pred = np.argmax(dnn_prob, axis=1)

# Ensemble
def pad(p, n):
    if p.shape[1] < n:
        r = np.zeros((p.shape[0], n)); r[:, :p.shape[1]] = p; return r
    return p

meta_X = np.hstack([pad(rf_prob, NUM_CLASSES), pad(xgb_prob, NUM_CLASSES), pad(dnn_prob, NUM_CLASSES)])
ens_pred = meta.predict(meta_X)
ens_prob = meta.predict_proba(meta_X)

# Baseline
baseline_pred = simulate_baseline(y_test)
print('All predictions computed!')

In [ ]:
# Compute all metrics
results = {}
for name, pred, prob in [('Baseline', baseline_pred, None), ('Random Forest', rf_pred, rf_prob),
                          ('XGBoost', xgb_pred, xgb_prob), ('DNN', dnn_pred, dnn_prob),
                          ('Ensemble', ens_pred, ens_prob)]:
    results[name] = compute_all_metrics(y_test, pred, prob, name)

# Summary table
summary = pd.DataFrame({
    name: {k: f'{v:.4f}' for k, v in m.items() if isinstance(v, float) and k in
           ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'fpr', 'detection_rate']}
    for name, m in results.items()
}).T
summary.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'FPR', 'Detection Rate']
print(summary.to_string())

In [ ]:
# Publication-ready comparison chart
metric_keys = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'detection_rate']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Detection Rate']

baseline_vals = [results['Baseline'][k] for k in metric_keys]
ensemble_vals = [results['Ensemble'][k] for k in metric_keys]

x = np.arange(len(display_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Wazuh Only (Baseline)',
               color='#e74c3c', alpha=0.85, edgecolor='white', linewidth=1.5)
bars2 = ax.bar(x + width/2, ensemble_vals, width, label='+ ML Ensemble (Enhanced)',
               color='#2ecc71', alpha=0.85, edgecolor='white', linewidth=1.5)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Intrusion Detection: Baseline vs. ML-Enhanced', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(display_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'models/baseline_vs_enhanced.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves for the ensemble
plot_roc_curves(y_test, ens_prob, 'Stacking Ensemble',
                os.path.join(project_root, 'models/ensemble_roc_curves.png'))
print('ROC curves saved!')

In [ ]:
# Improvement summary
print('\n' + '=' * 55)
print('  IMPROVEMENT SUMMARY (Baseline → Ensemble)')
print('=' * 55)
for metric in ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'detection_rate']:
    bv = results['Baseline'][metric]
    ev = results['Ensemble'][metric]
    diff = (ev - bv) * 100
    print(f'  {metric:<25s}: {bv:.4f} → {ev:.4f} (↑ {diff:+.1f}%)')
fpr_diff = (results['Baseline']['fpr'] - results['Ensemble']['fpr']) * 100
print(f"  {'FPR reduction':<25s}: {results['Baseline']['fpr']:.4f} → {results['Ensemble']['fpr']:.4f} (↓ {fpr_diff:.1f}%)")
print('=' * 55)